In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/config.json
/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/training_args.bin
/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/tokenizer.json
/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/tokenizer_config.json
/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/model.safetensors
/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model/generation_config.json
/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/competitions/deep-past-initiative-mac

In [2]:
import os
os.listdir("/kaggle/input/datasets/parlindchat")

['akkadian-model']

In [3]:
!pip install sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "/kaggle/input/datasets/parlindchat/akkadian-model/akkadian_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
import pandas as pd
test_df = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv")
test_df.head()

,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [5]:
batch_size = 16
predictions = []

for i in range(0, len(test_df), batch_size):

    batch_texts = [
        "translate Akkadian to English: " + t
        for t in test_df["transliteration"].iloc[i:i+batch_size].tolist()
    ]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=180,
        num_beams=5,
        length_penalty=1.1,
        repetition_penalty=2.0
    )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predictions.extend(preds)

In [6]:
print(len(predictions))
print(len(test_df))

4
4


In [7]:
test_df.head()

,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


In [8]:
submission = pd.DataFrame({ "id": test_df["id"], "translation": predictions })
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [9]:
import os
os.listdir("/kaggle/working")

['submission.csv', '__notebook__.ipynb']